# 1. LSTM - 长短时记忆网络 与 GRU - 门控网络

## 1.1 LSTM
$$
       \begin{align}
    i_t &= \sigma(W_{ii}x_t + b_{ii} + W_{hi}h_{t-1} + b_{hi}) \\
    f_t &= \sigma(W_{if}x_t + b_{if} + W_{hf}h_{t-1} + b_{hf}) \\
    g_t &= \tanh(W_{ig}x_t + b_{ig} + W_{hg}h_{t-1} + b_{hg}) \\
    o_t &= \sigma(W_{io}x_t + b_{io} + W_{ho}h_{t-1} + b_{ho}) \\
    c_t &= f_t \odot c_{t-1} + i_t \odot g_t \\
    h_t &= o_t \odot \tanh(c_t)
    \end{align}$$

## 1.2 GRU
$$
        \begin{align}
    r_t &= \sigma(W_{ir}x_t + b_{ir} + W_{hr}h_{t-1} + b_{hr}) \\
    z_t &= \sigma(W_{iz}x_t + b_{iz} + W_{hz}h_{t-1} + b_{hz}) \\
    n_t &= \tanh(W_{in}x_t + b_{in} + r_t \odot (W_{hn}h_{t-1} + b_{hn})) \\
    h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1}
    \end{align}$$

# 2. torch实现

- 模型
    - Embedding, RNN, LSTM, GRU
- 数据集
    - datasets加载
    - Qwen分词器 - BPE
        - 预训练分词
        - 定制BPE

In [1]:
from torch import nn
import torch

## 2.1 使用RNN、LSTM、GRU构建分类任务的网络模型

In [2]:
class RNNClassifierModel(nn.Module):

    def __init__(self,
                 vocab_size, # 词袋大小
                 embed_dim, # 向量化维度
                 hidden_dim, # 隐藏层维度
                 num_classes=5, # 分类数
                 model_type='lstm', # 特征层模型
                 num_layers=1, # 特征块层数
                 dropout=0.3, # 抛弃指数
                 device='cuda' # 设备
    ):
        super().__init__()
        self.device = device
        self.model_type = model_type
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        # layer_design
        ## 词嵌入
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        ## 特征
        if self.model_type == 'rnn':
            self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, num_layers=self.num_layers, dropout=dropout if self.num_layers > 1 else 0.0, batch_first=True, device=self.device)
        elif self.model_type == 'lstm':
            self.rnn = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, num_layers=self.num_layers, dropout=dropout if self.num_layers > 1 else 0.0, batch_first=True, device=self.device)
        elif self.model_type == 'gru':
            self.rnn = nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, num_layers=self.num_layers, dropout=dropout if self.num_layers > 1 else 0.0, batch_first=True, device=self.device)
        else:
            raise Exception('模型不匹配')
        ## 分类
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, hidden=None):
        """
        params:
            x: 输入的样本，维度 -> [batch_size, seq_len] -> embed后 -> [batch_size, seq_len, embed_dim]
            hidden: 初始化的隐藏特征
        return:
            output: 特征矩阵 -> [batch_size, seq_len, num_classes]
        """
        # 1. Embedding_model
        x = self.embedding(x)
        # 2. RNN_model
        if hidden is None:
            if self.model_type == 'lstm':
                h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(self.device)
                c_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(self.device)
                hidden = (h_0, c_0)
            else:
                hidden = h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(self.device)
        output, hidden = self.rnn(x, hidden)
        output = output.sum(dim=1)
        # 3. MPL_model
        output = self.fc(output)

        
        return output

In [44]:
x = torch.randint(0, 999, size=(16, 30))
model = RNNClassifierModel(1000, 1024, 128)
result = model(x)
result.shape

torch.Size([16, 5])

## 2.2 构建批次集

### 2.2.1 加载数据集

In [3]:
from datasets import load_dataset
# 数据集加载模块
ds = load_dataset(
    "csv",
    data_files={
        "train": "./datasets/train.csv",
        "vaild": "./datasets/test.csv"
    }
)

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'content'],
        num_rows: 4000
    })
    vaild: Dataset({
        features: ['label', 'content'],
        num_rows: 495
    })
})

In [5]:
ds["train"]

Dataset({
    features: ['label', 'content'],
    num_rows: 4000
})

In [6]:
ds["train"][0]

{'label': '体育',
 'content': '中国“铁腰”与英超球队埃弗顿分道扬镳，闪电般转投谢联（本赛季成功升入英超），此事运作速度之快令人惊诧。针对李铁与埃弗顿“分手”的原因、与埃弗顿主帅莫耶斯矛盾以及铁子为何选择谢联，记者昨日采访了李铁的母亲王桂芹，李母道出了李铁与埃弗顿分开的真实原因。龙菲坚决让铁子走人记者在采访王桂芹时了解到，李铁离开埃弗顿主要是妻子龙菲建议。龙菲平时不太过问李铁的足球方面事宜，但是，因为李铁长时间不能在埃弗顿踢上球，龙菲也十分焦急。多次安慰李铁后，龙菲想这样下去也不是个办法，于是索性做出决定，让李铁离开埃弗顿，只要能踢上球，去哪支球队都行。但前提条件必须是英国的球队。王妈妈告诉记者：“媳妇龙菲一直在英国学习，这孩子特别懂事，一边学习，一边还要照顾女儿和李铁的日常生活。对于李铁与埃弗顿的前前后后，龙菲一直都了解内情，因此龙菲最后告诉李铁，就是埃弗顿再请我们，我们也不去了，只要能离开埃弗顿，去哪支球队踢球都行。”据悉，龙菲２００１年便在沈阳拿到了留学英国全额奖学金的录取通知书，而后，龙菲便一直在英国求学。红牌让李铁失去位置“拼命三郎”、“跑不死”、“体能王”，这些溢美之词都是称赞李铁的，不过，正是因为李铁防守时的动作过于凶狠，在英超的赛场上屡次领到红牌。过多的红牌让主帅莫耶斯逐渐对李铁失去了兴趣。对此，王妈妈向记者表示：“关于球队的相关事宜，我不太清楚。因为埃弗顿主教练莫耶斯一直都很器重李铁，因为莫耶斯的战术比较偏重防守。而在防守过程中，李铁也确实吃到过红牌，但是，我觉得教练组和俱乐部不能因为红牌的原因不让李铁上场吧。”许宏涛挽救李铁李铁成功转会埃弗顿的整个过程，国内足球著名的经纪人许宏涛功不可没。而李铁在埃弗顿后期四处碰壁的危难时刻，正是许宏涛的左右逢源，令李铁还能坚持在埃弗顿预备队踢球。后来，在埃弗顿摒弃李铁后，也正是许宏涛的人脉关系让李铁再次找到了位置。根据李铁与埃弗顿签订的合同，今年６月３０日，工作合同才到期。但是，考虑到李铁与俱乐部的关系日益紧张，许宏涛便提早联系了英超其他球队，重点便是当时的英甲球队谢联。由于许宏涛是谢联董事会的成员，多次与俱乐部沟通李铁的事宜后，俱乐部最终同意了李铁加盟谢联。对李铁加盟谢联的事情，王桂芹不愿多谈。她只是表示：“许宏涛一直在帮助李铁，特别是在英国，许宏涛非常熟悉那里的环境，李铁也非常信任

In [8]:
ds["train"][0]['label']

'体育'

### 2.2.2 BPE分词器

In [4]:
from transformers import AutoTokenizer

In [5]:
# 加载预训练好的模型
tokenizer = AutoTokenizer.from_pretrained(r'D:\Qwen3.5-0.8B\Qwen3.5-0.8B', use_fast=True)

In [6]:
tokenizer

Qwen2TokenizerFast(name_or_path='D:\Qwen3.5-0.8B\Qwen3.5-0.8B', vocab_size=248044, model_max_length=262144, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>'], 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	248044: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	248045: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single

### 2.2.3 构建dataloader

#### 2.2.3.1 数据集

In [7]:
classes = ["体育", "军事", "健康", "教育", "汽车"]
class2id = {cls: idx for idx, cls in enumerate(classes)}

In [28]:
max_len = 1024

def token_fun(sample):
    """
    params:
        sample: {'label': .., 'content': ...}
    """
    tokenied = tokenizer(
        sample['content'],
        truncation=True,
        padding=False,
        max_length=max_len
    )

    tokenied['labels'] = [class2id[l] for l in sample['label']]

    return tokenied

In [29]:
tokenied_ds = ds.map(token_fun, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/495 [00:00<?, ? examples/s]

In [53]:
tokenied_ds

DatasetDict({
    train: Dataset({
        features: ['label', 'content', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    vaild: Dataset({
        features: ['label', 'content', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 495
    })
})

In [31]:
tokenied_ds.set_format(type="torch", columns=["input_ids", "labels"])

#### 2.2.3.2 批次集

In [32]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

In [35]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    max_length=max_len,
    padding="max_length",
    return_tensors="pt"
)

In [36]:
train_loader = DataLoader(tokenied_ds['train'], batch_size=32, shuffle=True, collate_fn=data_collator)
vaild_loader = DataLoader(tokenied_ds['vaild'], batch_size=32, shuffle=True, collate_fn=data_collator)

## 2.3 模型训练

In [45]:
from sklearn.metrics import accuracy_score

In [66]:
@torch.no_grad
def eval_model(model, dataloader):
    # 如果模型用了 -- Dropout、BatchNorm、layerNorm，要使用eval和train进行模式切换
    model.eval() # 评估模式
    model.to(model.device)
    predicts = [] # 预测标签
    actuals = [] # 真实标签
    for data in dataloader:
        outputs = model(data['input_ids'].to(model.device))

        prob, predicted = torch.max(outputs, dim=1) # prob 最大类别概率; predicted 
        predicts.extend(predicted.cpu().numpy())
        actuals.extend(data['labels'].cpu().numpy())
    return accuracy_score(actuals, predicts)

In [67]:
model = RNNClassifierModel(vocab_size=len(tokenizer.get_vocab()),
                           embed_dim=512,
                           hidden_dim=256,
                           num_classes=5,
                           model_type='lstm',
                           num_layers=1,
                           dropout=0.3,
                           device='cuda'
        )

epoch = 100
lr = 0.0005

loss_f = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)
model.to(model.device)

for e in range(epoch):
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        outputs = model(data['input_ids'].to(model.device))
        loss = loss_f(outputs, data['labels'].to(model.device))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.cpu().detach().item()
    avg_loss = total_loss / len(train_loader)
    accu = eval_model(model, vaild_loader)
    print(f"epoch: {e:03d}, avg_loss: {avg_loss:.6f}, accu: {accu * 100:.2f}%")
print("训练结束")

epoch: 000, avg_loss: 32.354217, accu: 70.91%
epoch: 001, avg_loss: 2.267164, accu: 85.86%
epoch: 002, avg_loss: 0.658446, accu: 91.31%
epoch: 003, avg_loss: 0.390026, accu: 86.26%
epoch: 004, avg_loss: 0.279345, accu: 81.62%
epoch: 005, avg_loss: 0.190586, accu: 87.88%
epoch: 006, avg_loss: 0.211702, accu: 91.52%
epoch: 007, avg_loss: 0.156759, accu: 89.49%
epoch: 008, avg_loss: 0.065821, accu: 92.12%
epoch: 009, avg_loss: 0.109891, accu: 90.71%
epoch: 010, avg_loss: 0.300675, accu: 86.67%
epoch: 011, avg_loss: 0.105548, accu: 93.13%
epoch: 012, avg_loss: 0.247253, accu: 93.74%
epoch: 013, avg_loss: 0.091597, accu: 89.70%
epoch: 014, avg_loss: 0.177704, accu: 92.73%
epoch: 015, avg_loss: 0.031482, accu: 90.71%
epoch: 016, avg_loss: 0.031793, accu: 94.55%
epoch: 017, avg_loss: 0.029236, accu: 94.14%
epoch: 018, avg_loss: 0.091252, accu: 94.34%
epoch: 019, avg_loss: 0.038822, accu: 91.72%
epoch: 020, avg_loss: 0.070274, accu: 93.94%
epoch: 021, avg_loss: 0.085236, accu: 95.35%
epoch: 02

### 保存

In [68]:
states = model.state_dict()
torch.save(states, "./models/rnn.pt")

## 2.4 测试与评估

In [16]:
def predict_text(text, model, tokenizer, max_len):
    model.to(model.device)
    model.eval()
    data = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )
    outputs = model(data['input_ids'].to(model.device))
    print(outputs.shape)
    # 这里搞一下
    outputs = torch.softmax(outputs, dim=1)
    prob, predicted = torch.max(outputs, dim=1)
    return prob.cpu().item(), predicted.cpu().item()

In [9]:
model = RNNClassifierModel(vocab_size=len(tokenizer.get_vocab()),
                           embed_dim=512,
                           hidden_dim=256,
                           num_classes=5,
                           model_type='lstm',
                           num_layers=1,
                           dropout=0.3,
                           device='cuda'
        )
# torch.load("./models/rnn.pt", map_location=model.device, weights_only=True) # 仅加载权重
model.load_state_dict(torch.load("./models/rnn.pt", map_location=model.device))
# model.to(model.device)
# model.eval()

<All keys matched successfully>

In [18]:
texts = [
    "届数比赛时间比赛地点参加国家和地区冠军亚军决赛成绩第一届1956-1957英国11美国丹麦6：1第二届1959-1960费城（美国）14美国英国5：2第三届1962-1963威尔明顿（美国）11美国英国4：3第四届1965-1966惠灵顿（新西兰）17日本美国5：2第五届1968-1969东京（日本）19日本印尼6：1第六届1971-1972东京（日本）17日本印尼6：1第七届1974-1975雅加达（印尼）14印尼日本5：2第八届1977-1978奥克兰（新西兰）16日本印尼5：2第九届1980-1981东京（日本）15日本印尼4：3第十届1983-1984吉隆坡（马来西亚）23中国印尼5：0第十一届1986雅加达（印尼）34中国印尼3：2第十二届1988吉隆坡（马来西亚）31中国韩国5：0第十三届1990东京（日本）42中国韩国3：2第十四届1992吉隆坡（马来西亚）44中国韩国3：2第十五届1994雅加达（印尼）44印尼中国3：2第十六届1996香港（中国）50印尼中国4：1第十七届1998香港（中国）50中国印尼5：0第十八届2000吉隆坡（马来西亚）50中国印尼5：0第十九届2002广州（中国）50中国韩国3：1第二十届2004雅加达（印尼）50中国韩国3：1第二十一届2006东京（日本）50中国荷兰3：0",
    "新华网华盛顿2月3日电(记者杨晴川潘云召)美国国防部3日在其网站上发布国防指导性文件《四年防务评估报告》，对美国武装力量面临的威胁进行了重新定义，将战略重点从常规战争转向恐怖主义、大规模杀伤性武器和新兴战略对手等3个新领域。美国国防部长拉姆斯菲尔德在报告前言中说，这份报告有4个重点，即打败极端主义、保卫美国本土、对付处于“战略十字路口”的新兴军事强国、防止危险政权和恐怖分子获得大规模杀伤性武器。这份长达92页的报告说，为应付来自3个新领域的威胁，美军将相应地扩充特种部队以打击恐怖主义和增强本土防御，加强应对大规模杀伤性武器袭击的能力并发展高科技常规武器遏制战略对手。报告特别强调反恐的重要性，并称反恐将是一场“长期战争”。报告提出了加强特种部队的许多建议，包括将特别行动司令部下属部队人数增加15%，陆军特种营数目增加33%，在海军陆战队内部新设特种行动司令部，空军设立无人航空器大队，增加海军特种部队人数，将特别行动司令部下属的心理战和民事部队人数增加33%等等。在应对大规模杀伤性武器方面，报告委派美军战略司令部建立一个应对大规模杀伤性武器威胁的快速反应联合行动总部。报告同时宣布美国军方将在未来5年内投入15亿美元，发展对付生化武器袭击的技术。为保持美军对于新兴战略对手的优势，报告呼吁发展包括水下发射远程导弹和无人飞机等一系列新式常规武器，同时强调保持强大的核威慑力量。今年的报告是美国宣布发动反恐战争后出台的首份《四年防务评估报告》，其中战略重点的调整反应了“9·11”事件后反恐战争和国际形势的变化对于美国国防策略的影响。美国防部副部长英格兰在前言中说，这一报告将加快美军自2001年开始的从冷战战略模式向应对非对称挑战和威胁转变的进程。在国会要求下，美国国防部从1997年起每4年出台一份《四年防务评估报告》，以此作为美国国防政策和军事战略的指导文件。今年的报告将于6日正式送交国会",
    "广东农村中小学教师可以享受公务员津贴和补贴，工资和福利也将纳入地方政府财政预算，这一举措如能顺利实施，无疑说明政府对农村中小学教师待遇问题的重视终于以实际行动表现了出来，自然令人鼓舞。不过，此次提高农村教师待遇的资金出处，并未点名是由谁来“埋单”。资金出处的语焉不详，表明这笔钱有可能还是由县、乡等基层财政来“埋单”，从历史上看，这也是以往“提高教师待遇”的惯例做法——即上级给政策、下级给资金。实践了多年的这种“提高教师待遇”的模式，到今天看来不能不让人心存疑虑。因为农村教师收入低恰恰就是因为农村地区财政支付能力较弱。城市与乡村巨大的经济差距，以及由此所导致的财政汲取能力差距，正是农村教师的收入与待遇低于城市教师的最重要原因。并因此导致了克扣农村教师工资、要求农村教师为各种项目捐款、摊派等非正常现象的大量出现。也就是说，农村基层财政本来就缺乏资金，如果只给政策，一些地方可能陷入“巧妇难为无米之炊”之境。而且，免征农业税之后的农村财政愈加捉襟见肘，事实上，农村教师们最担心的问题正是怕地方财政拿不出钱来——非不为也，是不能也。从根本上说，真正提高教师待遇的途径无外以下几种:一、打破教育垄断，吸收更多的社会资金和社会资源、乃至于志愿组织进入义务教育领域，使得教育经费的来源多元化；二，各级政府加大教育投入，增加公共服务性支出在整个财政支出中的比例，至少要达到国际平均水平；三、破除导致城乡差别的种种限制，改善农村、特别是基层农村的财政状况。当然，这些都涉及到政治和社会体制的结构性改革，并非一朝一夕之功。在现有的条件下，我们所能做的只能是由上级财政通过转移支付来弥补城乡教师待遇上的差距。毕竟，一项严肃的政策出台，其初衷决不是为了“既赢得尊师重教的美名，又不用花一分钱”。广东这几年来在教育投入、教师待遇提高、解决贫困学生问题等方面成绩斐然，省级财政也为此做出了巨大贡献。我们有理由期待，今次“提高教师待遇”政策的出台，其配套措施、特别是资金来源的问题，应已纳入考量范围。(编辑:苏琳)",
    "据英国广播公司（BBC）等媒体报道，美国五角大楼当地时间7日公布了41份不明飞行物（UFO）相关文件，包括十多个视频、三张图片和数十份PDF格式文档。这是自今年5月以来，美国国防部应总统特朗普要求公布的第五批相关文件。据报道描述，其中一段视频由美国军方摄像头于2025年拍摄，显示一个“神秘球体”在中东地区一个似乎是居民区的上空快速移动；还有一段视频于2019年拍摄，显示一个黑色球体悬浮在太平洋上空，似乎短暂消失后又重新出现，并缓慢地从左向右漂移。另有一段于2021年在阿曼湾拍摄的视频画面似乎显示，一个黑色的“冷球”以不规则模式移动。该物体粗看像一只飞掠过画面的家蝇，但据报道，这是一个直径约4英尺（1.2米）的“不明异常现象（UAP）”物体。",
    "据《图片报》报道，卡里乌斯讲述了自己最黑暗的时期，如今他在沙尔克04焕发新生。"
]

for text in texts:
    prob, predicted = predict_text(text, model, tokenizer, max_len=1024)
    print(f"置信度:{prob}, 预测类别:{classes[predicted]}")

torch.Size([1, 5])
置信度:1.0, 预测类别:体育
torch.Size([1, 5])
置信度:1.0, 预测类别:军事
torch.Size([1, 5])
置信度:1.0, 预测类别:教育
torch.Size([1, 5])
置信度:0.9999825954437256, 预测类别:教育
torch.Size([1, 5])
置信度:0.4784098267555237, 预测类别:教育


In [ ]:
for data in vaild_loader:
    with torch.no_grad():
        outputs = model(data['input_ids'].to(model.device))
        print(outputs.shape)
        _, predicted = torch.max(outputs, dim=1)
        for i in range(len(predicted)):
            print(f"预测结果: {predicted[i]}, 真实结果: {data['labels'][i]}")